In [ ]:
import random
import asyncio
import re
import pandas as pd
from vpei.utils.llm_requests_v3 import *
from vpei.utils.llm_utils import save_model_experimental_results_to_csv
from vpei.common_utils import extract_score, extract_string
from vpei.common_variables import *
from vpei.epistemic_consistency.experiment_utils import *
from vpei.epistemic_consistency.experiment_types import carry_out_comparative_experiment_with_ground_truth_and_multiple_choices
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS

# function_to_perturb_solution = randomly_flip_digits
function_to_perturb_solution = randomly_flip_single_digit

df = pd.read_csv("./data/sample_physics_problems_and_solutions.csv")
df.rename(columns={"problem": "physics_problem"}, inplace=True)
df['physics_solution'] = df['answers'] + " " + df['unit'].fillna('')
df['perturbed_physics_solution'] = df['physics_solution'].apply(function_to_perturb_solution)

# For comparative with ground truth: correct vs incorrect (perturbed) solutions
# We create two dataframes with matching structure but different solution columns
df_correct = df.copy()
df_correct.drop(columns=['perturbed_physics_solution'], inplace=True)
df_incorrect = df.copy()
df_incorrect['physics_solution'] = df_incorrect['perturbed_physics_solution']
df_incorrect.drop(columns=['perturbed_physics_solution'], inplace=True)
df_incorrect

In [ ]:
experiment_name = "physics_problems"
number_of_choices = 5
system_prompt = EXPERIMENTS[experiment_name]["comparative_experiment_with_ground_truth_and_multiple_choices"]["system_prompt"]
user_prompt_template_repeated_block = EXPERIMENTS[experiment_name]["comparative_experiment_with_ground_truth_and_multiple_choices"]["user_prompt_template_repeated_block"]
user_prompt_template_repeated_attribution_block = EXPERIMENTS[experiment_name]["comparative_experiment_with_ground_truth_and_multiple_choices"]["user_prompt_template_repeated_attribution_block"]
print(system_prompt)
print("-------------------------------------------------------------------")
print(build_user_prompt_template_with_variable_repeats(number_of_choices, user_prompt_template_repeated_block, user_prompt_template_repeated_attribution_block))

In [ ]:
model_name = "gpt-5-mini"
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})

user_prompt_template = build_user_prompt_template_with_variable_repeats(number_of_choices, user_prompt_template_repeated_block, user_prompt_template_repeated_attribution_block)

names_dict = {
"name_1": "P.P.",
"name_2": "A.J.",
"name_3": "L.K.",
"name_4": "M.R.",
"name_5": "S.T.",
}
political_attitudes = {
"political_attitude_1": "conservative",
"political_attitude_2": "progressive",
"political_attitude_3": "left-leaning",
"political_attitude_4": "right-wing",
"political_attitude_5": "Republican",
}

stimuli_factors_into_user_prompt = {
"physics_problem_1": df_incorrect.iloc[0]['physics_problem'],
"physics_solution_1": df_incorrect.iloc[0]['physics_solution'],
"physics_problem_2": df_incorrect.iloc[1]['physics_problem'],
"physics_solution_2": df_incorrect.iloc[1]['physics_solution'],
"physics_problem_3": df_incorrect.iloc[2]['physics_problem'],
"physics_solution_3": df_incorrect.iloc[2]['physics_solution'],
"physics_problem_4": df_correct.iloc[3]['physics_problem'],
"physics_solution_4": df_correct.iloc[3]['physics_solution'],
"physics_problem_5": df_incorrect.iloc[4]['physics_problem'],
"physics_solution_5": df_incorrect.iloc[4]['physics_solution'],
}

user_prompt = user_prompt_template.format(**names_dict,
                                                      **political_attitudes,
                                                      **stimuli_factors_into_user_prompt)


messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
make_llm_request(model_name, messages, **model_kwargs)

In [ ]:
models = ["gpt-5-mini"]


n = 5
custom_model_kwargs = {}
stimuli_factors = ["physics_problem", "physics_solution"]
additional_variables_from_df_to_save = [] 
number_of_choices = 5
path_to_save_model_outputs = "./comparative_experiment_with_ground_truth_and_multiple_choices"
random_seed = 42

In [ ]:
payloads = await carry_out_comparative_experiment_with_ground_truth_and_multiple_choices(models=models, df_correct=df_correct, df_incorrect=df_incorrect, n=n, 
                                                                                         system_prompt=system_prompt, user_prompt_template_repeated_block=user_prompt_template_repeated_block, user_prompt_template_repeated_attribution_block=user_prompt_template_repeated_attribution_block,
                                                                                         stimuli_factors=stimuli_factors, additional_variables_from_df_to_save=additional_variables_from_df_to_save,
                                                                                         custom_model_kwargs=custom_model_kwargs, path_to_save_model_outputs=path_to_save_model_outputs,
                                                                                         random_seed=21, number_of_choices=number_of_choices)

df = pd.DataFrame(payloads)
df['model_response_pole'].value_counts()